# 第二章：教会模型看懂图文 — CLIP 对比学习

> 上一章，我们的 ViT 只能做图像分类——它只知道"猫"或"狗"，
> 不理解"一只在草地上奔跑的橘猫"这样的自然语言描述。
> 本章解决这个问题。

---

## 本章目标

从零实现 **CLIP**（Contrastive Language-Image Pre-training），理解：

1. 为什么分类预训练不够用，需要图文对齐
2. 双编码器架构：图像和文本各一个 Transformer
3. **InfoNCE 对比损失**：从数学原理到代码实现
4. 温度参数的作用
5. 用合成数据训练并验证对齐效果
6. CLIP 最神奇的能力：零样本图像分类

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), '.'))
os.makedirs('figures', exist_ok=True)

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

---
## 2.1 为什么图像分类预训练不够？

上一章的 ViT 在 ImageNet 1000 类上预训练后，能识别猫、狗、汽车……
但这样的特征有一个根本局限：

**特征空间是封闭的**——只有 1000 个离散标签，
无法描述"一只正在打哈欠的猫"和"一只在睡觉的猫"的区别，
更无法回答"图里有几只猫"这样的问题。

CLIP 的思路是：**不用标签，用自然语言本身作为监督信号**。

训练数据只需要图文对：`(图像, 对应的描述文字)`
—— 互联网上有几十亿对这样的数据，获取成本极低。

训练目标很直接：
**让图像和它对应的文字描述在嵌入空间中靠近，不对应的远离。**

In [ ]:
from multimodal_from_scratch.figures import draw_clip_architecture
fig = draw_clip_architecture(save_path='figures/ch02_clip_arch.png')
plt.show()

---
## 2.2 InfoNCE 对比损失：从直觉到公式

### 2.2.1 直觉

假设一个 batch 有 N 对图文：`{(I₁,T₁), (I₂,T₂), ..., (Iₙ,Tₙ)}`

- **正样本对**：`(Iᵢ, Tᵢ)` — 图像和它对应的文字（N 对，对角线）
- **负样本对**：`(Iᵢ, Tⱼ)` where `i ≠ j` — 不匹配的组合（N²−N 对）

对每张图像 `Iᵢ`，我们希望模型能从 N 个文本中找到正确的 `Tᵢ`。
这其实就是一个 **N 分类问题**！

### 2.2.2 数学形式

设图像嵌入和文本嵌入都经过 L2 归一化，相似度矩阵为：

```
S[i,j] = f(Iᵢ) · g(Tⱼ) / τ
```

其中 `τ`（tau）是**温度参数**，控制分布的尖锐程度。

InfoNCE 损失（对称版本）：

```
L = (1/2) * [
    image→text:  (1/N) * Σᵢ -log( exp(S[i,i]) / Σⱼ exp(S[i,j]) )
  + text→image:  (1/N) * Σⱼ -log( exp(S[j,j]) / Σᵢ exp(S[i,j]) )
]
```

注意：这就是对相似度矩阵的**行和列同时做 CrossEntropy**，目标是让**对角线**最大。

### 2.2.3 为什么叫"对比"损失？

InfoNCE 对嵌入空间同时施加两种力：
- **引力**：把匹配的图文对拉近（正样本）
- **斥力**：把不匹配的图文对推远（负样本）

这两种力的平衡，使嵌入空间形成有意义的语义结构。

In [ ]:
# 用手动方式一步步理解 InfoNCE，再封装成函数

torch.manual_seed(0)
N = 3
# 模拟 3 对图文嵌入（已经 L2 归一化）
img_emb = F.normalize(torch.randn(N, 4), dim=-1)   # (3, 4)
txt_emb = F.normalize(torch.randn(N, 4), dim=-1)   # (3, 4)

tau = 0.07  # CLIP 原文使用的温度

# 相似度矩阵：S[i,j] = 第 i 张图与第 j 段文字的相似度
sim = img_emb @ txt_emb.T / tau   # (3, 3)
print("相似度矩阵 S = img_emb @ txt_emb.T / tau:")
print(sim.detach().numpy().round(2))

# 正确标签：对角线索引 [0, 1, 2]
labels = torch.arange(N)
print(f"\n正确匹配 (对角线) 的索引: {labels.tolist()}")

# 图像→文本方向：每行的正确列（对应文本）应该最大
loss_i2t = F.cross_entropy(sim, labels)
# 文本→图像方向：每列的正确行（对应图像）应该最大
loss_t2i = F.cross_entropy(sim.T, labels)

loss = (loss_i2t + loss_t2i) / 2
print(f"\n图像→文本 loss: {loss_i2t.item():.4f}")
print(f"文本→图像 loss: {loss_t2i.item():.4f}")
print(f"InfoNCE loss:   {loss.item():.4f}")
print(f"随机猜测下限:   {np.log(N):.4f}  (= ln({N})，随机时的期望 loss)")

In [ ]:
# 封装成可复用的函数
def infonce_loss(img_emb: torch.Tensor, txt_emb: torch.Tensor,
                 temperature: float = 0.07) -> torch.Tensor:
    """
    对称 InfoNCE（CLIP）对比损失。

    Args:
        img_emb:     (B, D)  已 L2 归一化的图像嵌入
        txt_emb:     (B, D)  已 L2 归一化的文本嵌入
        temperature: τ，越小分布越尖锐（更难的负样本学习）
    Returns:
        标量损失
    """
    B = img_emb.shape[0]
    sim = img_emb @ txt_emb.T / temperature   # (B, B)
    labels = torch.arange(B, device=img_emb.device)
    return (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2


# 验证：完全对齐的嵌入 loss 应该接近 0
perfect_emb = F.normalize(torch.eye(4), dim=-1)   # 4 个正交向量，完全不重叠
loss_perfect = infonce_loss(perfect_emb, perfect_emb)
loss_random  = infonce_loss(
    F.normalize(torch.randn(4, 8), dim=-1),
    F.normalize(torch.randn(4, 8), dim=-1)
)
print(f"完全对齐时 loss: {loss_perfect.item():.4f}  （接近 0）")
print(f"随机嵌入时 loss: {loss_random.item():.4f}   （接近 ln(4)={np.log(4):.2f}）")

### 2.2.4 温度参数 τ 的作用

τ 控制相似度分布的"尖锐程度"：

| τ 值 | 效果 | 问题 |
|------|------|------|
| 很大（如 2.0） | 分布平坦，所有 token 概率相近 | 学习信号弱，模型分不清正负样本 |
| 适中（0.07）  | 正样本概率明显高于负样本 | CLIP 原文的默认值 |
| 很小（如 0.01）| 分布极尖，最难的负样本主导梯度 | 训练不稳定，容易梯度爆炸 |

CLIP 将 τ 设为**可学习参数**（以 log 形式存储保证正值），
训练开始时 τ = 0.07，模型会自动调整到最优值。

In [ ]:
# 可视化不同温度下的 softmax 分布
logits_raw = torch.tensor([2.0, 0.5, 0.3, -0.1])   # 一个 query 对 4 个 key 的相似度

temps = [0.01, 0.07, 0.5, 2.0]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))

for ax, tau in zip(axes, temps):
    probs = F.softmax(logits_raw / tau, dim=0).numpy()
    bar_colors = ['#2ECC71' if i == 0 else '#E74C3C' for i in range(4)]
    ax.bar(range(4), probs, color=bar_colors, edgecolor='white', linewidth=1.2)
    ax.set_xticks(range(4))
    ax.set_xticklabels([f'T{i+1}' for i in range(4)])
    ax.set_ylim(0, 1.05)
    ax.set_title(f'tau = {tau}', fontsize=12, fontweight='bold')
    ax.set_ylabel('softmax 概率' if tau == temps[0] else '')
    ax.axhline(0.25, color='grey', linestyle='--', alpha=0.5, lw=1)
    ax.text(0, probs[0]+0.03, f'{probs[0]:.2f}', ha='center', fontsize=9,
            color='#27AE60', fontweight='bold')

axes[0].text(1.5, 0.88, '绿色=正样本\n红色=负样本', fontsize=8,
             bbox=dict(facecolor='lightyellow', edgecolor='grey', boxstyle='round'))
plt.suptitle('温度参数 tau 对 softmax 分布的影响', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/ch02_temperature.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 2.3 对比学习的直觉：嵌入空间的"引力与斥力"

InfoNCE 的目标可以形象地理解为：在嵌入空间中，
同一概念的图像表示和文字表示应该"住在一起"，不同概念应该"相距甚远"。

In [ ]:
from multimodal_from_scratch.figures import draw_infonce_intuition
fig = draw_infonce_intuition(save_path='figures/ch02_infonce.png')
plt.show()

---
## 2.4 实现 CLIP 模型

CLIP 由两个编码器组成：
- **图像编码器**：上一章的 ViT（取 [CLS] token）
- **文本编码器**：一个双向 Transformer，取 [EOS] token 的输出

两个编码器的输出通过各自的**投影层**映射到相同维度 `proj_dim`，
然后 L2 归一化后计算点积相似度。

> 注意：文本编码器和 GPT 不同——它是**双向**的（没有因果掩码），
> 因为我们只需要理解文本，不需要生成文本。

In [ ]:
# 复用第1章的基础组件
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x): return self.proj(x).flatten(2).transpose(1, 2)

class PositionalEmbedding(nn.Module):
    def __init__(self, n_patches, embed_dim):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
    def forward(self, x): return x + self.pos_embed

class BidirectionalAttention(nn.Module):
    """双向多头自注意力（ViT 和 CLIP 文本编码器均使用此版本）"""
    def __init__(self, embed_dim, num_heads, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads; self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(0)
        attn = F.softmax((q @ k.transpose(-2,-1)) * self.scale, dim=-1)
        return self.out_proj((self.drop(attn) @ v).transpose(1,2).reshape(B,N,C))

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim); self.attn = BidirectionalAttention(embed_dim, num_heads, dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        h = int(embed_dim * mlp_ratio)
        self.ffn = nn.Sequential(nn.Linear(embed_dim, h), nn.GELU(), nn.Linear(h, embed_dim), nn.Dropout(dropout))
    def forward(self, x):
        x = x + self.attn(self.norm1(x)); return x + self.ffn(self.norm2(x))

print("基础组件定义完毕 ✓")

In [ ]:
class CLIPImageEncoder(nn.Module):
    """图像编码器：ViT backbone + 线性投影"""

    def __init__(self, img_size, patch_size, embed_dim, depth, num_heads, proj_dim):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, 3, embed_dim)
        n = self.patch_embed.n_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = PositionalEmbedding(n, embed_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        # 将 ViT 的 embed_dim 投影到共享空间 proj_dim
        self.proj = nn.Linear(embed_dim, proj_dim, bias=False)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(B,-1,-1), x], dim=1)
        x = self.norm(self.blocks(self.pos_embed(x)))
        # 取 [CLS] token，投影 + L2 归一化
        return F.normalize(self.proj(x[:, 0]), dim=-1)


class CLIPTextEncoder(nn.Module):
    """
    文本编码器：双向 Transformer + 线性投影。
    取 [EOS] token 位置的输出作为整段文字的表示。

    为什么用 [EOS] 而不是 [CLS]？
    CLIP 的文本编码器没有专门的 [CLS] token，
    而是在序列末尾放 [EOS]（end-of-sequence），
    经过双向注意力后，[EOS] 能汇聚整段文字的语义。
    """

    def __init__(self, vocab_size, context_len, embed_dim, depth, num_heads, proj_dim):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed   = nn.Embedding(context_len, embed_dim)
        self.blocks = nn.Sequential(*[TransformerBlock(embed_dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.proj = nn.Linear(embed_dim, proj_dim, bias=False)

    def forward(self, input_ids, eos_pos):
        """
        input_ids: (B, T)  token 序列
        eos_pos:   (B,)    每个序列中 [EOS] token 的位置索引
        """
        B, T = input_ids.shape
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        x = self.token_embed(input_ids) + self.pos_embed(pos)
        x = self.norm(self.blocks(x))
        # 取 [EOS] 位置，投影 + L2 归一化
        eos_feat = x[torch.arange(B), eos_pos]
        return F.normalize(self.proj(eos_feat), dim=-1)


class CLIP(nn.Module):
    """
    CLIP：双编码器 + 可学习温度的 InfoNCE 损失。

    训练后可用于：
    1. 图文匹配（相似度打分）
    2. 零样本图像分类
    3. 作为 VLM 的视觉编码器（第3章）
    """

    def __init__(self, img_size=32, patch_size=8,
                 vision_dim=64, vision_depth=2, vision_heads=4,
                 vocab_size=256, context_len=16,
                 text_dim=64, text_depth=2, text_heads=4,
                 proj_dim=64):
        super().__init__()
        self.image_encoder = CLIPImageEncoder(
            img_size, patch_size, vision_dim, vision_depth, vision_heads, proj_dim)
        self.text_encoder = CLIPTextEncoder(
            vocab_size, context_len, text_dim, text_depth, text_heads, proj_dim)
        # 可学习温度：log(1/0.07) ≈ 2.66，即初始 tau = 0.07
        self.logit_scale = nn.Parameter(torch.tensor(2.6592))

    def encode_image(self, images):
        return self.image_encoder(images)

    def encode_text(self, input_ids, eos_pos):
        return self.text_encoder(input_ids, eos_pos)

    def forward(self, images, input_ids, eos_pos):
        img_emb = self.encode_image(images)           # (B, proj_dim)
        txt_emb = self.encode_text(input_ids, eos_pos)  # (B, proj_dim)

        tau = self.logit_scale.exp().clamp(max=100)
        sim = img_emb @ txt_emb.T * tau               # (B, B)

        labels = torch.arange(img_emb.shape[0], device=img_emb.device)
        loss = (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels)) / 2

        return {'loss': loss, 'sim': sim, 'img_emb': img_emb, 'txt_emb': txt_emb}


# 快速验证
clip = CLIP()
images  = torch.randn(4, 3, 32, 32)
tokens  = torch.randint(0, 256, (4, 16))
eos_pos = torch.tensor([14, 14, 14, 14])

out = clip(images, tokens, eos_pos)
print(f"Loss: {out['loss'].item():.3f}  （随机时期望: {np.log(4):.3f}）")
print(f"相似度矩阵形状: {out['sim'].shape}  (B x B)")
print(f"当前温度 tau = {clip.logit_scale.exp().item():.3f}")

---
## 2.5 训练演示：让颜色和文字对齐

### 合成数据集设计

我们构造一个极简的图文对齐任务：
- **图像**：4 种纯色（红/蓝/绿/黄）+ 随机噪声
- **文本**：对应颜色的英文名（"red", "blue", "green", "yellow"）
- **评估指标**：相似度矩阵的对角线是否明显高于非对角线

这个任务虽然简单，但完整展示了 CLIP 的学习机制。
真实的 CLIP 在 4 亿个更复杂的图文对上做同样的事。

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def make_color_clip_dataset(n_per_class=100, img_size=32, context_len=16, seed=0):
    """
    4 类颜色图文对数据集。
    文本用最简单的字符级 tokenizer（ASCII 码 % 128）。
    """
    torch.manual_seed(seed)
    color_rgb   = [[1.0,0.1,0.1], [0.1,0.1,1.0], [0.1,0.8,0.1], [0.9,0.8,0.0]]
    color_words = ['red', 'blue', 'green', 'yellow']

    all_imgs, all_tokens, all_eos, all_labels = [], [], [], []

    for label, (rgb, word) in enumerate(zip(color_rgb, color_words)):
        char_ids = [ord(c) % 128 for c in word]
        eos_pos  = len(char_ids)
        padded   = char_ids + [1] + [0] * (context_len - len(char_ids) - 1)

        for _ in range(n_per_class):
            img = torch.tensor(rgb).reshape(3,1,1).expand(3, img_size, img_size).clone()
            img += torch.randn_like(img) * 0.06
            all_imgs.append(img.clamp(0, 1))
            all_tokens.append(torch.tensor(padded, dtype=torch.long))
            all_eos.append(eos_pos)
            all_labels.append(label)

    return (torch.stack(all_imgs), torch.stack(all_tokens),
            torch.tensor(all_eos), torch.tensor(all_labels))

imgs, tokens, eos_positions, labels = make_color_clip_dataset()
print(f"图像: {imgs.shape},  token: {tokens.shape},  类别: {labels.unique().tolist()}")

dataset = TensorDataset(imgs, tokens, eos_positions, labels)
loader  = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
# 训练 CLIP
clip = CLIP(img_size=32, patch_size=8, vocab_size=128, context_len=16, proj_dim=32)
optimizer = torch.optim.AdamW(clip.parameters(), lr=3e-4, weight_decay=0.01)

losses = []
clip.train()
for epoch in range(20):
    epoch_loss = 0
    for xb_img, xb_tok, xb_eos, _ in loader:
        out = clip(xb_img, xb_tok, xb_eos)
        optimizer.zero_grad(); out['loss'].backward(); optimizer.step()
        epoch_loss += out['loss'].item()
    avg = epoch_loss / len(loader)
    losses.append(avg)
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/20 | loss={avg:.4f} | tau={clip.logit_scale.exp().item():.3f}")

In [ ]:
# 可视化训练结果
clip.eval()
color_names = ['red', 'blue', 'green', 'yellow']

with torch.no_grad():
    rep_imgs   = torch.stack([imgs[labels == i][0] for i in range(4)])
    rep_tokens = torch.stack([tokens[labels == i][0] for i in range(4)])
    rep_eos    = torch.tensor([int(eos_positions[labels == i][0]) for i in range(4)])
    img_emb = clip.encode_image(rep_imgs)
    txt_emb = clip.encode_text(rep_tokens, rep_eos)
    sim_matrix = (img_emb @ txt_emb.T).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(range(1, 21), losses, 'b-o', markersize=5)
axes[0].axhline(np.log(4), color='red', linestyle='--', alpha=0.6,
                label=f'随机基线 ln(4)={np.log(4):.2f}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('InfoNCE Loss')
axes[0].set_title('CLIP 训练损失', fontsize=12, fontweight='bold')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

im = axes[1].imshow(sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1, aspect='equal')
axes[1].set_xticks(range(4)); axes[1].set_yticks(range(4))
axes[1].set_xticklabels(color_names); axes[1].set_yticklabels(color_names)
axes[1].set_xlabel('文本嵌入'); axes[1].set_ylabel('图像嵌入')
axes[1].set_title('训练后相似度矩阵\n（对角线应最高）', fontsize=12, fontweight='bold')
for i in range(4):
    for j in range(4):
        axes[1].text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                     fontsize=10, fontweight='bold',
                     color='white' if abs(sim_matrix[i,j]) > 0.5 else 'black')
plt.colorbar(im, ax=axes[1])
plt.tight_layout()
plt.savefig('figures/ch02_clip_trained.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 2.6 零样本分类：CLIP 最神奇的能力

训练完 CLIP 后，我们可以**无需任何标注数据**做图像分类——
只需把类别名称变成文本嵌入，再找相似度最高的那个。

这在 2021 年震惊了 AI 社区：
一个从未见过 ImageNet 标签的模型，在零样本设置下达到了接近监督训练的准确率。

原理极其简单：
```python
predicted_class = argmax( image_embedding @ class_text_embeddings.T )
```

In [ ]:
def zero_shot_classify(clip_model, image, class_words, context_len=16):
    """零样本分类：用类别文字嵌入的相似度直接预测。"""
    clip_model.eval()
    with torch.no_grad():
        img_emb = clip_model.encode_image(image.unsqueeze(0))  # (1, D)

        text_embs = []
        for word in class_words:
            char_ids = [ord(c) % 128 for c in word]
            eos_pos  = len(char_ids)
            padded   = char_ids + [1] + [0] * (context_len - len(char_ids) - 1)
            ids = torch.tensor(padded, dtype=torch.long).unsqueeze(0)
            eos = torch.tensor([eos_pos])
            text_embs.append(clip_model.encode_text(ids, eos))
        text_embs = torch.cat(text_embs, dim=0)   # (n_classes, D)

        scores = (img_emb @ text_embs.T).squeeze()
        probs  = F.softmax(scores * 10, dim=-1)
    return probs

# 测试零样本分类
class_names = ['red', 'blue', 'green', 'yellow']
print(f"{'真实类别':<10} {'red':>8} {'blue':>8} {'green':>8} {'yellow':>8} {'预测':>8}")
print('-' * 58)
for true_label, color in enumerate(class_names):
    test_img = imgs[labels == true_label][5]
    probs = zero_shot_classify(clip, test_img, class_names)
    pred  = class_names[probs.argmax().item()]
    mark  = 'v' if pred == color else 'x'
    row   = f"{color:<10}"
    for p in probs: row += f" {p.item():>7.1%}"
    print(row + f"  {pred} {mark}")

---
## 本章小结

### 我们构建了什么

| 组件 | 作用 | 关键点 |
|------|------|--------|
| `infonce_loss()` | 图文对比目标函数 | 对角线 CrossEntropy，对称双向 |
| `CLIPImageEncoder` | 图像 → 嵌入向量 | ViT + 投影层 + L2 归一化 |
| `CLIPTextEncoder` | 文本 → 嵌入向量 | 双向 Transformer，取 [EOS] 输出 |
| `CLIP` | 图文对齐模型 | 可学习温度参数 |

### 关键洞察

**1. Batch size 即负样本数量**
更大的 batch → 更多负样本 → 更难的任务 → 更好的特征。
这是为什么 CLIP 用了 32,768 的超大 batch size，需要数百块 GPU 同步训练。

**2. 温度参数是训练稳定性的关键**
太小的 τ 会让梯度主要来自最难的几个负样本，
一旦这些负样本恰好是语义相近的样本（比如两张不同的猫），
会产生错误的梯度方向。可学习 τ 让模型自适应调整。

**3. CLIP 学的是"关系"，不是"类别"**
模型没有 1000 个输出节点，它学到的是嵌入空间的几何结构——
语义相近的图文在空间中距离近，语义无关的距离远。
这使它能泛化到从未见过的概念。

### 下一章预告

CLIP 给了我们高质量的图像编码器。
但它只能"看图匹配文字"，不能"看图生成文字"。
下一章，我们用一个小小的 Projection MLP 把 ViT 和 GPT 连起来，
构建第一个能对话的 VLM。